# Weather steering: reproduction and baselines

In [ ]:
AGENT = "ollama"  # or "anthropic" (needs ANTHROPIC_API_KEY in Kaggle secrets)
OLLAMA_MODEL = "qwen2.5:7b"
ANTHROPIC_MODEL = "claude-haiku-4-5"
REPEATS = 5
UPSTREAM_COMMIT = "13faf5214f7cbea07d005278f1aeaf3973611eba"
DECIDER = "StrandsAgents/strands-decider-2B-qwen3.5-v1-2610"

In [ ]:
import os, glob, json, time, subprocess, urllib.request

!rm -rf /kaggle/working/strands-decider
!git clone -q https://github.com/strands-labs/strands-decider /kaggle/working/strands-decider
!git -C /kaggle/working/strands-decider checkout -q {UPSTREAM_COMMIT}
!pip install -q -e /kaggle/working/strands-decider
!pip uninstall -y -q torchao  # old torchao on Kaggle breaks peft

pkg = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
       if 'name = "steering-baselines"' in open(p).read()][0]
!rm -rf /kaggle/working/steering-baselines && cp -r {pkg} /kaggle/working/steering-baselines
!pip install -q -e /kaggle/working/steering-baselines

In [ ]:
def wait_for(url, proc, log, tries):
    for _ in range(tries):
        if proc.poll() is not None:
            raise RuntimeError(open(log).read()[-3000:])
        try:
            return urllib.request.urlopen(url, timeout=5).read()
        except Exception:
            time.sleep(10)
    raise RuntimeError(f"timed out waiting for {url}")

log = "/kaggle/working/decider.log"
decider = subprocess.Popen(["strands-decider", "serve", DECIDER, "--device", "cpu", "--port", "8099"],
                           stdout=open(log, "w"), stderr=subprocess.STDOUT)
print(json.loads(wait_for("http://127.0.0.1:8099/health", decider, log, 180)))

In [ ]:
if AGENT == "ollama":
    !apt-get install -y -qq zstd > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh
    log = "/kaggle/working/ollama.log"
    ollama = subprocess.Popen(["ollama", "serve"], stdout=open(log, "w"), stderr=subprocess.STDOUT)
    wait_for("http://127.0.0.1:11434/api/version", ollama, log, 30)
    !ollama pull {OLLAMA_MODEL}
    MODEL = f"ollama:{OLLAMA_MODEL}"
else:
    from kaggle_secrets import UserSecretsClient
    os.environ["ANTHROPIC_API_KEY"] = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
    MODEL = f"anthropic:{ANTHROPIC_MODEL}"
MODEL

## Reproduction

In [ ]:
!cd /kaggle/working && python -m steering_baselines.run_weather --gates upstream --cases upstream --model {MODEL} --repeats {REPEATS} --out runs/reproduction

## Baselines

In [ ]:
!cd /kaggle/working && python -m steering_baselines.run_weather --gates never,always,upstream,threshold,llm --model {MODEL} --repeats {REPEATS} --out runs/baselines

In [ ]:
!cd /kaggle/working && zip -qr results.zip runs decider.log